# ArSL Word Training — Kaggle GPU (KArSL-502 Image Sequence Edition)

**Independent Mode** — No external vocabulary files needed.  
Trained directly from the KArSL-502 image-frame dataset on Kaggle.

### Dataset Format
This notebook is built for the KArSL-502 dataset where each sign sample is stored as a **folder of `.jpg` frames** (not `.mp4` videos).

```
KARSL-502/{class_id}/{class_id}/{train|test}/{sample_id}/{recording_folder}/
    frame_001.jpg
    frame_002.jpg
    ...
```


In [ ]:
# CELL 1: IMPORTS + MEDIAPIPE TASKS SETUP
# MediaPipe 0.10.10+ uses the Tasks API (no 'solutions' module).
# The hand_landmarker.task model is loaded from a Kaggle dataset input.

import os, sys, time, warnings
from pathlib import Path

import cv2, numpy as np, pandas as pd, tensorflow as tf
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision

warnings.filterwarnings('ignore')

# ---- Locate hand_landmarker.task model ----
# Upload hand_landmarker.task to Kaggle as a dataset named: hand-landmarker-model
# It will appear at: /kaggle/input/hand-landmarker-model/hand_landmarker.task
MODEL_PATH = None

if os.path.exists('/kaggle'):
    # Scan all /kaggle/input/ datasets for the .task file
    for root, dirs, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname.endswith('.task'):
                MODEL_PATH = os.path.join(root, fname)
                break
        if MODEL_PATH:
            break
else:
    # Local fallback
    local = Path(r'M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\hand_landmarker.task')
    if local.exists():
        MODEL_PATH = str(local)

print(f'Model path : {MODEL_PATH if MODEL_PATH else "NOT FOUND"}')

# ---- Import MediaPipe Tasks ----
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

MEDIAPIPE_AVAILABLE = MODEL_PATH is not None and os.path.exists(MODEL_PATH)

print(f'TensorFlow : {tf.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'MediaPipe  : {mp.__version__}')
print(f'Model ready: {MEDIAPIPE_AVAILABLE}')
print('All imports OK!' if MEDIAPIPE_AVAILABLE else 'ERROR: hand_landmarker.task not found — see instructions below.')

if not MEDIAPIPE_AVAILABLE:
    print()
    print('FIX: Upload hand_landmarker.task to Kaggle as a dataset:')
    print('  1. Go to kaggle.com > Datasets > + New Dataset')
    print('  2. Name it: hand-landmarker-model')
    print('  3. Upload: hand_landmarker.task')
    print('  4. Add it to this notebook via + Add Input')


In [ ]:
# =========================
# CELL 2: GPU SETUP
# =========================
print('=' * 60)
print('GPU DETECTION')
print('=' * 60)

gpus = tf.config.list_physical_devices('GPU')
USE_GPU = False
DEVICE = '/CPU:0'

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        USE_GPU = True
        DEVICE = '/GPU:0'
        print(f'GPU AVAILABLE: {gpus[0].name}')
    except RuntimeError as e:
        print(f'GPU error: {e}')
else:
    print('No GPU — training on CPU (slower)')

mixed_precision.set_global_policy('float32')
print(f'Using device: {DEVICE}')
print('=' * 60)


In [ ]:
# =========================
# CELL 3: CONFIGURATION
# =========================

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    KAGGLE_INPUT = Path('/kaggle/input')
    KAGGLE_OUTPUT = Path('/kaggle/working')
    OUTPUT_DIR = KAGGLE_OUTPUT

    # === AUTO-DETECT KARSL ROOT ===
    # Kaggle datasets can be at various paths depending on how they're added.
    # We try multiple common locations.
    KARSL_ROOT = None
    candidates = [
        KAGGLE_INPUT / 'karsl-502',
        KAGGLE_INPUT / 'KARSL-502',
        KAGGLE_INPUT / 'karsl502',
    ]
    # Also check if there's a subfolder inside (some datasets have an extra wrapper)
    for c in list(candidates):
        candidates.append(c / 'KARSL-502')
        candidates.append(c / 'karsl-502')

    # Also scan everything directly in /kaggle/input/
    if KAGGLE_INPUT.exists():
        for item in os.scandir(str(KAGGLE_INPUT)):
            if item.is_dir():
                candidates.append(Path(item.path))
                # Check if there's a nested folder
                for sub in os.scandir(item.path):
                    if sub.is_dir():
                        candidates.append(Path(sub.path))

    # Find the one that has numbered subfolders (01, 02, etc.)
    for c in candidates:
        if not c.exists():
            continue
        try:
            has_class_folders = any(
                d.is_dir() and d.name.isdigit()
                for d in os.scandir(str(c))
            )
            if has_class_folders:
                KARSL_ROOT = c
                break
        except:
            continue

    if KARSL_ROOT is None:
        # Last resort: print what's actually in /kaggle/input/ for debugging
        print('Could not auto-detect KArSL root. Contents of /kaggle/input/:')
        for item in os.scandir(str(KAGGLE_INPUT)):
            print(f'  {item.name} ({"dir" if item.is_dir() else "file"})')
            if item.is_dir():
                for sub in os.scandir(item.path):
                    print(f'    {sub.name} ({"dir" if sub.is_dir() else "file"})')
        raise FileNotFoundError('Cannot find KArSL dataset. Check Input panel.')
else:
    PROJECT_ROOT = Path(r'M:/Term 10/Grad')
    KARSL_ROOT = PROJECT_ROOT / 'SLR Main/Words/Datasets/KArSL_502'
    OUTPUT_DIR = PROJECT_ROOT / 'SLR Main/Words/ArSL Word (Arabic)'

# Independent mode uses class IDs directly, so initialize fallback label maps.
id_to_english = {}
id_to_arabic = {}

# ===== PARAMETERS =====
SEQUENCE_LENGTH = 30
NUM_HANDS = 2
LANDMARKS_PER_HAND = 63    # 21 landmarks x 3
NUM_FEATURES = NUM_HANDS * LANDMARKS_PER_HAND  # 126

BATCH_SIZE      = 64
EPOCHS          = 150
LEARNING_RATE   = 5e-4
LSTM_UNITS_1    = 256
LSTM_UNITS_2    = 128
LSTM_UNITS_3    = 64
DENSE_UNITS     = 256
DROPOUT_RATE    = 0.4
LABEL_SMOOTH    = 0.1
TEST_SIZE       = 0.4

OUTPUT_DIR.mkdir(parents=True, exist_ok=True) if not IS_KAGGLE else None

print(f'KArSL root      : {KARSL_ROOT}')
print(f'Output dir      : {OUTPUT_DIR}')
print(f'Sequence length : {SEQUENCE_LENGTH}')
print(f'Features/frame  : {NUM_FEATURES}')
print(f'Batch size      : {BATCH_SIZE}')
print(f'Max epochs      : {EPOCHS}')
print(f'Running on      : {"Kaggle" if IS_KAGGLE else "Local"}')


In [ ]:
# CELL 4: DISCOVER REAL CLASS IDs & BUILD RECORDING MAP
# KArSL structure: KARSL_ROOT/group/group/{test,train}/CLASS_ID/recording/
# The top-level folders (01, 02, 03) are GROUPS, not class IDs.
# The real class IDs are 4-digit folders inside test/train splits.

print('=' * 60)
print('DISCOVERING REAL CLASS STRUCTURE')
print('=' * 60)

if not KARSL_ROOT.exists():
    raise FileNotFoundError(f'KArSL dataset not found: {KARSL_ROOT}')

# Build: class_id (int) -> list of recording folder paths
class_recordings = {}

group_dirs = sorted([
    e for e in os.scandir(str(KARSL_ROOT))
    if e.is_dir() and e.name.isdigit()
], key=lambda e: e.name)

print(f'Top-level group folders: {len(group_dirs)} ({[g.name for g in group_dirs[:5]]}...)')

for group_entry in group_dirs:
    # Level 1: group folder (e.g. 01/)
    for inner_entry in os.scandir(group_entry.path):
        if not inner_entry.is_dir():
            continue
        # Level 2: inner folder (same name, e.g. 01/01/)
        for split_entry in os.scandir(inner_entry.path):
            if not split_entry.is_dir():
                continue
            # Level 3: test/ or train/ split
            for class_entry in os.scandir(split_entry.path):
                if not class_entry.is_dir() or not class_entry.name.isdigit():
                    continue
                # Level 4: real class ID folder (e.g. 0111)
                class_id = int(class_entry.name)
                if class_id not in class_recordings:
                    class_recordings[class_id] = []
                # Level 5: recording folders (contain .jpg frames)
                try:
                    for rec_entry in os.scandir(class_entry.path):
                        if rec_entry.is_dir():
                            class_recordings[class_id].append(rec_entry.path)
                except:
                    pass

# Sort class IDs
class_ids = sorted(class_recordings.keys())
target_karsl_classes = class_ids

# Fill in label mappings
for cid in class_ids:
    if cid not in id_to_english:
        id_to_english[cid] = str(cid)
    if cid not in id_to_arabic:
        id_to_arabic[cid] = str(cid)

total_recs = sum(len(v) for v in class_recordings.values())

print(f'\nReal class IDs found   : {len(class_ids)}')
print(f'Total recording folders: {total_recs}')
print(f'Avg recordings/class   : {total_recs / max(len(class_ids), 1):.1f}')
print(f'\nSample classes:')
for cid in class_ids[:10]:
    n = len(class_recordings[cid])
    print(f'  Class {cid:4d} ({id_to_english[cid]:20s}) : {n} recordings')


In [ ]:
# CELL 5: HELPER FUNCTIONS (MediaPipe Tasks API)

SEQUENCE_LENGTH = 30
LANDMARKS_PER_HAND = 63
NUM_FEATURES = 2 * LANDMARKS_PER_HAND  # 126

def pad_or_sample(sequence, target_len=SEQUENCE_LENGTH, target_features=NUM_FEATURES):
    arr = np.array(sequence, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[0] == 0:
        return None
    if arr.shape[1] > target_features:
        arr = arr[:, :target_features]
    elif arr.shape[1] < target_features:
        pad = np.zeros((arr.shape[0], target_features - arr.shape[1]), dtype=np.float32)
        arr = np.concatenate([arr, pad], axis=1)
    if arr.shape[0] >= target_len:
        idx = np.linspace(0, arr.shape[0] - 1, target_len, dtype=int)
        arr = arr[idx]
    else:
        pad = np.zeros((target_len - arr.shape[0], target_features), dtype=np.float32)
        arr = np.concatenate([arr, pad], axis=0)
    return arr


def make_hand_detector():
    if MODEL_PATH is None or not os.path.exists(MODEL_PATH):
        raise RuntimeError('hand_landmarker.task not found. Re-run Cell 1.')
    options = HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return HandLandmarker.create_from_options(options)


def extract_landmarks_from_frame(rgb_frame, detector):
    left_vec  = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
    right_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = detector.detect(mp_image)
        if result.hand_landmarks:
            for hand_lms, handedness in zip(result.hand_landmarks, result.handedness):
                label = handedness[0].category_name
                vec = np.array([[lm.x, lm.y, lm.z] for lm in hand_lms]).flatten()
                if label == 'Left':
                    left_vec = vec
                else:
                    right_vec = vec
    except Exception:
        pass
    return np.concatenate([left_vec, right_vec])


def extract_from_image_folder_2hand(folder_path, detector, img_files=None):
    # img_files can be pre-sorted list of paths (avoids repeated scandir)
    if img_files is None:
        try:
            img_files = sorted([
                e.path for e in os.scandir(str(folder_path))
                if e.is_file() and e.name.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
        except:
            return None
    if len(img_files) < 3:
        return None
    frames_data = []
    for img_path in img_files:
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        vec = extract_landmarks_from_frame(rgb, detector)
        frames_data.append(vec)
    if len(frames_data) < 3:
        return None
    return pad_or_sample(np.array(frames_data, dtype=np.float32))


print('Helper functions ready (MediaPipe Tasks API)')
print(f'NUM_FEATURES = {NUM_FEATURES}  ({LANDMARKS_PER_HAND} per hand x 2 hands)')


In [ ]:
# CELL 6: BUILD DATASET (or Load Cached)

print('=' * 60)
print('BUILDING ARABIC WORD DATASET')
print('=' * 60)

NPZ_PATH = OUTPUT_DIR / 'arsl_word_sequences_2hand.npz'

if NPZ_PATH.exists():
    print(f'\nCached data found: {NPZ_PATH}')
    data = np.load(NPZ_PATH)
    X, y = data['X'], data['y']
    print(f'   X shape : {X.shape}')
    print(f'   y shape : {y.shape}')
    print(f'   Classes : {len(np.unique(y))}')
    print('   Loaded from cache — skipping extraction')
else:
    if not MEDIAPIPE_AVAILABLE:
        raise RuntimeError('MediaPipe model not available. Re-run Cell 1.')

    print(f'\nInitializing MediaPipe hand detector...')
    detector = make_hand_detector()
    print('Detector ready.\n')

    # ---- SANITY CHECK on first class ----
    _first_id = target_karsl_classes[0]
    _first_recs = class_recordings[_first_id]
    _first_rec = _first_recs[0]

    print(f'Sanity check — Class {_first_id} ({id_to_english[_first_id]})')
    print(f'  Recordings      : {len(_first_recs)}')
    print(f'  First recording : {os.path.basename(_first_rec)}')

    _imgs = sorted([
        e.path for e in os.scandir(_first_rec)
        if e.is_file() and e.name.lower().endswith(('.jpg','.jpeg','.png'))
    ])
    print(f'  Images in rec   : {len(_imgs)}')

    if _imgs:
        _f = cv2.imread(_imgs[0])
        if _f is not None:
            _rgb = cv2.cvtColor(_f, cv2.COLOR_BGR2RGB)
            _vec = extract_landmarks_from_frame(_rgb, detector)
            _nz  = np.count_nonzero(_vec)
            print(f'  Frame shape     : {_f.shape}')
            print(f'  Feature vec     : {_vec.shape}, non-zero={_nz}/{len(_vec)}')
            print(f'  Hand detected   : {"YES" if _nz > 0 else "NO (ok if blurry)"}')

    print('Sanity check done — starting full extraction\n')
    print('=' * 60)

    # ---- FULL EXTRACTION ----
    print(f'Extracting {len(target_karsl_classes)} classes | {sum(len(v) for v in class_recordings.values())} total recordings')
    print(f'(showing every recording for first 2 classes, then per-class summary)\n')

    start_time = time.time()
    X_list, y_list = [], []
    found_classes, empty_classes = 0, 0
    total_proc, skipped = 0, 0
    VERBOSE_CLASSES = 2  # show live per-recording for first N classes

    for ci, class_id in enumerate(target_karsl_classes):
        recordings = class_recordings.get(class_id, [])
        if not recordings:
            empty_classes += 1
            continue

        found_classes += 1
        class_ok = 0
        label = id_to_english.get(class_id, str(class_id))

        if ci < VERBOSE_CLASSES:
            print(f'\n[Class {ci+1}/{len(target_karsl_classes)}] ID={class_id} "{label}" — {len(recordings)} recordings')

        for rec_path in recordings:
            total_proc += 1
            rec_name = os.path.basename(rec_path)

            # Count images
            try:
                imgs = sorted([
                    e.path for e in os.scandir(rec_path)
                    if e.is_file() and e.name.lower().endswith(('.jpg','.jpeg','.png'))
                ])
            except:
                skipped += 1
                continue

            if ci < VERBOSE_CLASSES:
                # Show every recording live
                seq = extract_from_image_folder_2hand(rec_path, detector, img_files=imgs)
            else:
                seq = extract_from_image_folder_2hand(rec_path, detector, img_files=imgs)

            if seq is None:
                skipped += 1
                if ci < VERBOSE_CLASSES:
                    print(f'  SKIP {rec_name[:50]} ({len(imgs)} imgs) -> None')
                continue

            blank_ratio = np.sum(np.all(seq == 0, axis=1)) / len(seq)
            if blank_ratio > 0.8:
                skipped += 1
                if ci < VERBOSE_CLASSES:
                    print(f'  SKIP {rec_name[:50]} ({len(imgs)} imgs) -> blank={blank_ratio:.2f}')
                continue

            X_list.append(seq)
            y_list.append(class_id)
            class_ok += 1

            if ci < VERBOSE_CLASSES:
                print(f'  OK   {rec_name[:50]} ({len(imgs)} imgs) blank={blank_ratio:.2f}')

        # Per-class summary line (always shown)
        elapsed = time.time() - start_time
        rate = (ci + 1) / elapsed if elapsed > 0 else 0
        eta = (len(target_karsl_classes) - ci - 1) / rate if rate > 0 else 0
        print(f'[{ci+1:3d}/{len(target_karsl_classes)}] {label[:20]:20s} | ok={class_ok:4d}/{len(recordings):4d} | '
              f'Total={len(X_list):6d} | {elapsed/60:.1f}m elapsed | ETA {eta/60:.1f}m')

    detector.close()

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)

    elapsed = time.time() - start_time
    print(f'\nDone in {elapsed/60:.1f} min')
    print(f'   X shape       : {X.shape}')
    print(f'   Classes found : {found_classes} / {len(target_karsl_classes)}')
    print(f'   Skipped       : {skipped} / {total_proc}')

    np.savez_compressed(NPZ_PATH, X=X, y=y)
    print(f'\nSaved: {NPZ_PATH}')


In [ ]:
# =========================
# CELL 7: DATA EXPLORATION
# =========================
print('=' * 60)
print('DATA EXPLORATION')
print('=' * 60)

if 'X' not in dir() or 'y' not in dir():
    data = np.load(NPZ_PATH)
    X, y = data['X'], data['y']

unique_ids, counts = np.unique(y, return_counts=True)
labels = [str(uid) for uid in unique_ids]

sort_idx = np.argsort(counts)[::-1]
sorted_names  = [labels[i] for i in sort_idx]
sorted_counts = counts[sort_idx]

fig, ax = plt.subplots(figsize=(22, 6))
ax.bar(range(len(sorted_names)), sorted_counts, color='darkgreen', edgecolor='black', linewidth=0.3)
ax.set_xticks(range(len(sorted_names)))
ax.set_xticklabels(sorted_names, rotation=90, fontsize=5)
ax.set_xlabel('Class ID', fontsize=12)
ax.set_ylabel('Samples', fontsize=12)
ax.set_title(f'ArSL Class Distribution — {len(unique_ids)} classes, {len(y)} total samples', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'class_distribution.png'), dpi=150)
plt.show()

print(f'\nTotal samples    : {len(y)}')
print(f'Total classes    : {len(unique_ids)}')
print(f'Avg samples/class: {len(y)/len(unique_ids):.1f}')
print(f'Min samples      : {counts.min()} (class {unique_ids[np.argmin(counts)]})')
print(f'Max samples      : {counts.max()} (class {unique_ids[np.argmax(counts)]})')


In [ ]:
# =========================
# CELL 8: PREPROCESSING & SPLIT
# =========================
print('=' * 60)
print('PREPROCESSING & SPLIT')
print('=' * 60)

data = np.load(NPZ_PATH)
X, y = data['X'], data['y']

# StandardScaler
original_shape = X.shape
X_flat = X.reshape(-1, NUM_FEATURES)
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(original_shape).astype(np.float32)

# Save scaler stats
np.savez_compressed(
    str(OUTPUT_DIR / 'arsl_scaler_stats.npz'),
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32)
)
print('Scaler saved')

# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_onehot = to_categorical(y_encoded, num_classes=num_classes)

# Save class mapping WITH REAL LABEL NAMES
classes_df = pd.DataFrame({
    'model_class_index': range(num_classes),
    'label_name': [id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)],
    'arabic_name': [id_to_arabic.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)],
    'source_class_id': [int(c) for c in encoder.classes_]
})
classes_df.to_csv(str(OUTPUT_DIR / 'arsl_word_classes.csv'), index=False)
print(f'Class mapping saved ({num_classes} classes) with English labels')
print(classes_df.head(10).to_string())

# Stratified split 60/20/20
try:
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_onehot, test_size=TEST_SIZE, random_state=42, stratify=y_encoded
    )
    temp_labels = np.argmax(y_temp, axis=1)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=temp_labels
    )
except ValueError:
    # Some classes may have too few samples for stratification
    print('WARNING: Falling back to non-stratified split')
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_onehot, test_size=TEST_SIZE, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42
    )

print(f'\nTrain : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')
print(f'Classes: {num_classes}')


In [ ]:
# =========================
# CELL 9: BUILD & TRAIN BiLSTM
# =========================
print('=' * 60)
print('TRAINING BiLSTM MODEL')
print('=' * 60)

tf.keras.backend.clear_session()

# Build model
model = Sequential([
    Bidirectional(
        LSTM(LSTM_UNITS_1, return_sequences=True),
        input_shape=(SEQUENCE_LENGTH, NUM_FEATURES)
    ),
    BatchNormalization(),
    Dropout(DROPOUT_RATE),

    Bidirectional(LSTM(LSTM_UNITS_2, return_sequences=True)),
    BatchNormalization(),
    Dropout(DROPOUT_RATE),

    LSTM(LSTM_UNITS_3, return_sequences=False),
    BatchNormalization(),
    Dropout(DROPOUT_RATE),

    Dense(DENSE_UNITS, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax', dtype='float32')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
    ]
)

model.summary()

# Callbacks
MODEL_BEST = str(OUTPUT_DIR / 'arsl_word_lstm_model_best.h5')
MODEL_FINAL = str(OUTPUT_DIR / 'arsl_word_lstm_model_final.h5')

callbacks = [
    ModelCheckpoint(MODEL_BEST, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, verbose=1, min_lr=1e-6),
]

# Balanced class weights
train_labels = np.argmax(y_train, axis=1)
class_weights_arr = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(class_weights_arr))

print(f'\nTraining with class weights (balanced)')
print(f'Batch size: {BATCH_SIZE}')
print(f'Max epochs: {EPOCHS}')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

model.save(MODEL_FINAL)
print(f'\nSaved best  : {MODEL_BEST}')
print(f'Saved final : {MODEL_FINAL}')


In [ ]:
# =========================
# CELL 10: EVALUATION
# =========================
print('=' * 60)
print('MODEL EVALUATION')
print('=' * 60)

# Load best model
best_model = tf.keras.models.load_model(MODEL_BEST)

# Predict
proba = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(proba, axis=1)
y_true = np.argmax(y_test, axis=1)

# Top-1 accuracy
top1_acc = (y_pred == y_true).mean()

# Top-5 accuracy
top5_correct = 0
for i in range(len(y_true)):
    top5 = np.argsort(proba[i])[-5:]
    if y_true[i] in top5:
        top5_correct += 1
top5_acc = top5_correct / len(y_true)

print(f'\nTop-1 Accuracy : {top1_acc*100:.2f}%')
print(f'Top-5 Accuracy : {top5_acc*100:.2f}%')

# Classification report
word_labels = [id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)]
print('\nClassification Report:')
print(classification_report(y_true, y_pred, labels=range(num_classes),
                            target_names=word_labels, zero_division=0))

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'ArSL Training — Top-1: {top1_acc*100:.1f}% | Top-5: {top5_acc*100:.1f}%',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(cm, annot=False, cmap='Greens',
            xticklabels=word_labels, yticklabels=word_labels, ax=ax)
ax.set_title(f'Confusion Matrix — {num_classes} classes')
plt.xticks(rotation=90, fontsize=4)
plt.yticks(fontsize=4)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'confusion_matrix.png'), dpi=150)
plt.show()


In [ ]:
# =========================
# CELL 11: OUTPUT FILES
# =========================
print('=' * 60)
print('OUTPUT FILES')
print('=' * 60)

if IS_KAGGLE:
    print('\nFiles available for download in /kaggle/working/:')
else:
    print(f'\nFiles saved to: {OUTPUT_DIR}')

for f in sorted(OUTPUT_DIR.glob('arsl_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'   {f.name} ({size_mb:.2f} MB)')

print('\nDownload these and place in your local ArSL Word (Arabic) folder.')
print('Then run ArSL_Word_Live_Test.ipynb to test with your webcam!')


## Troubleshooting

| Issue | Fix |
|-------|-----|
| **OOM** | Reduce `BATCH_SIZE` to 32 or 16 in Cell 3 |
| **No GPU** | Settings → Accelerator → GPU T4 |
| **Slow extraction** | Normal — takes 1-3 hours for full dataset |
| **NaN loss** | Reduce `LEARNING_RATE` to 1e-4 |
| **Low accuracy** | Increase `EPOCHS`, reduce number of classes |
| **Dataset not found** | Cell 3 will print what's in /kaggle/input/ for debugging |
